In [ ]:
# papermill parameters cell — injected at runtime
sample_id = ""
session_dir = ""
out_dir = ""
thickness_nm = 2.5
area_by_size = {"big": 771786, "mid": 192180, "small": 67400, "little": 28508, "tiny": 6333}
cal_freq_hz = 1e4
title = ""

In [ ]:
import json
import math
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EPS0 = 8.854e-12  # F/m

out_path = Path(out_dir)
out_path.mkdir(parents=True, exist_ok=True)
session_path = Path(session_dir)

plot_title = title if title else sample_id
print(f"Session: {session_dir}")
print(f"Output : {out_dir}")
print(f"d={thickness_nm} nm  area_by_size={area_by_size}")

In [ ]:
# Discover all devices that have CF sweep data
# Device folder = session_dir/<device_name>/
# CF file:  <device>/<device>_run_<ts>/data/*_cf_0V_*frequency_log*.csv
# CV file:  same run dir          /data/*_cv_100kHz_*bias_linear*.csv

SIZE_PREFIXES = list(area_by_size.keys())  # big, mid, small, little, tiny

def size_from_name(device_name):
    """Extract size prefix; handle littlel typo."""
    raw = device_name.split('_')[0]
    if raw == 'littlel':
        return 'little'
    return raw if raw in area_by_size else None

devices = []  # list of dicts: {name, size, area_um2, radius_um, cf_file, cv_file}

for dev_dir in sorted(session_path.iterdir()):
    if not dev_dir.is_dir():
        continue
    name = dev_dir.name
    size = size_from_name(name)
    if size is None:
        print(f"  SKIP {name}: unknown size prefix")
        continue

    # Find CF file
    cf_files = sorted(dev_dir.glob('*/data/*_cf_*frequency_log*.csv'))
    cv_files = sorted(dev_dir.glob('*/data/*_cv_*bias_linear*.csv'))

    if not cf_files:
        print(f"  SKIP {name}: no CF file found")
        continue

    area_um2 = area_by_size[size]
    r_um = math.sqrt(area_um2 / math.pi)

    devices.append({
        'name': name,
        'size': size,
        'area_um2': area_um2,
        'radius_um': r_um,
        'cf_file': cf_files[0],
        'cv_file': cv_files[0] if cv_files else None,
    })

print(f"Found {len(devices)} CF/CV devices:")
for d in devices:
    print(f"  {d['name']} (size={d['size']}, r={d['radius_um']:.1f} um)")

In [ ]:
# Load CF and CV data for each device
d_m = thickness_nm * 1e-9

for dev in devices:
    A_m2 = dev['area_um2'] * 1e-12

    # --- CF (C vs frequency at 0V) ---
    df_cf = pd.read_csv(dev['cf_file'])
    # drop the 0-frequency sentinel row (sweep_value=0 at start)
    df_cf = df_cf[df_cf['set_frequency_Hz'] > 0].copy()
    df_cf = df_cf[df_cf['C_F'].notna() & (df_cf['C_F'] > 0)].copy()
    freq = df_cf['measured_frequency_Hz'].to_numpy(dtype=float)
    C = df_cf['C_F'].to_numpy(dtype=float)
    k = C * d_m / (EPS0 * A_m2)
    dev['freq'] = freq
    dev['C_pF'] = C * 1e12
    dev['k'] = k

    # k at cal_freq
    idx = int(np.argmin(np.abs(freq - cal_freq_hz)))
    dev['k_at_cal'] = float(k[idx])
    dev['freq_at_cal'] = float(freq[idx])

    # --- CV (C vs bias at 100kHz) ---
    dev['bias'] = None
    dev['C_cv_pF'] = None
    if dev['cv_file'] is not None:
        df_cv = pd.read_csv(dev['cv_file'])
        df_cv = df_cv[df_cv['C_F'].notna()].copy()
        dev['bias'] = df_cv['set_bias_V'].to_numpy(dtype=float)
        dev['C_cv_pF'] = df_cv['C_F'].to_numpy(dtype=float) * 1e12

    print(f"{dev['name']}: CF rows={len(freq)}, k@cal={dev['k_at_cal']:.3f}")

In [ ]:
# Group by size for bold median trace
sizes_present = sorted(set(d['size'] for d in devices),
                        key=lambda s: -area_by_size[s])  # big -> tiny

# Color palette — one per size group
PALETTE = plt.rcParams['axes.prop_cycle'].by_key()['color']
size_color = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(sizes_present)}

def median_device(group):
    """Pick device whose k_at_cal is nearest the group median."""
    ks = np.array([d['k_at_cal'] for d in group])
    med = np.median(ks)
    return group[int(np.argmin(np.abs(ks - med)))]

# 3-panel layout: top row = C(F) + k(F); bottom = k vs radius (full width)
fig = plt.figure(figsize=(14, 10))
ax_cf = fig.add_subplot(2, 2, 1)   # panel 1: C(F)
ax_kf = fig.add_subplot(2, 2, 2)   # panel 2: k(F)
ax_kr = fig.add_subplot(2, 1, 2)   # panel 3: k vs radius (spans full bottom row)

legend_handles = []

for size in sizes_present:
    group = [d for d in devices if d['size'] == size]
    color = size_color[size]
    r_um = group[0]['radius_um']
    label = f"r={r_um:.1f} um"
    rep = median_device(group)

    for dev in group:
        # Faint individual lines
        ax_cf.semilogx(dev['freq'], dev['C_pF'], color=color, alpha=0.25, linewidth=0.8)
        ax_kf.semilogx(dev['freq'], dev['k'], color=color, alpha=0.25, linewidth=0.8)

    # Bold median line + markers
    h, = ax_cf.semilogx(rep['freq'], rep['C_pF'], color=color, linewidth=2.0,
                         marker='.', markersize=5, label=label)
    ax_kf.semilogx(rep['freq'], rep['k'], color=color, linewidth=2.0,
                   marker='.', markersize=5, label=label)
    legend_handles.append(h)

ax_cf.set_xlabel('Frequency F (Hz)')
ax_cf.set_ylabel('C (pF)')
ax_cf.set_title('1) C(F)')
ax_cf.legend(fontsize=8, ncol=2)
ax_cf.grid(True, which='both', alpha=0.3)

ax_kf.set_xlabel('Frequency F (Hz)')
ax_kf.set_ylabel('k')
ax_kf.set_title('2) k(F)')
ax_kf.set_ylim(1.0, 2.5)
ax_kf.grid(True, which='both', alpha=0.3)

# --- Panel 3: k vs radius at cal_freq ---
all_k_at_cal = [d['k_at_cal'] for d in devices]
mean_k_global = float(np.mean(all_k_at_cal))

for size in sizes_present:
    group = [d for d in devices if d['size'] == size]
    color = size_color[size]
    r_um = group[0]['radius_um']
    ks = np.array([d['k_at_cal'] for d in group])
    mean_k = float(np.mean(ks))
    std_k = float(np.std(ks)) if len(ks) > 1 else 0.0

    # Faint individual points
    for d in group:
        ax_kr.scatter([r_um], [d['k_at_cal']], color=color, alpha=0.4, s=20, zorder=3)

    # Mean±std errorbar
    marker = {'big': 'D', 'mid': '^', 'small': 's', 'little': 'd', 'tiny': 'D'}.get(size, 'o')
    ax_kr.errorbar([r_um], [mean_k], yerr=[std_k], fmt=marker,
                   color=color, markersize=9, capsize=4, linewidth=1.5, zorder=5)

ax_kr.axhline(mean_k_global, color='gray', linestyle=':', linewidth=1.2)
ax_kr.text(0.02, 0.95, f'\u27e8k\u27e9 = {mean_k_global:.2f}',
           transform=ax_kr.transAxes, va='top', fontsize=10)
ax_kr.set_xlabel('Electrode radius r (um)')
ax_kr.set_ylabel('k')
ax_kr.set_title(f'3) k(V) at {cal_freq_hz:.0e} Hz')
ax_kr.grid(True, alpha=0.3)

fig.suptitle(plot_title, fontsize=12, y=1.01)
fig.tight_layout()

out_png = out_path / 'cv_cf_summary.png'
fig.savefig(str(out_png), dpi=130, format='png', bbox_inches='tight')
plt.close(fig)
print(f"Saved: {out_png}")


In [ ]:
# Compute per-size mean k and write metrics.json
k_by_size = {}
for size in sizes_present:
    group = [d for d in devices if d['size'] == size]
    k_by_size[size] = float(np.mean([d['k_at_cal'] for d in group]))

metrics = {
    'mean_k': mean_k_global,
    'k_by_size': k_by_size,
    'n_devices': len(devices),
    'cal_freq_hz': cal_freq_hz,
    'thickness_nm': thickness_nm,
}

metrics_path = out_path / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics written to {metrics_path}")
print(json.dumps(metrics, indent=2))